In [1]:
import pandas as pd
import torch
import numpy as np
from torch_geometric.data import Data
from biopandas.pdb import PandasPdb
import matplotlib.pyplot as plt

In [2]:
df = pd.read_csv(f'expt0_train.csv')

In [3]:
layer=31
mdl = 'esm2_t33_650M_UR50D'
### emb_raw_pth: location where the ESM embeddings are stored
### see https://github.com/facebookresearch/ESM for embedding ESM extractoin 
emb_raw_pth = f'extract_esm2_t33_650M_UR50D_layer{layer}' 
emb_ext_pth = f'{mdl}_layer{layer}_feat_v1_CBTree_cleanV1_split/'

In [ ]:
for index, row in df.iterrows():
    pdbpth = row['pth'] ### location of the stored PDB file. If no PDB is available, use the Uni_resid for the Uniprot sequence and pos_in_seq should be uni_resid - 1
    resid = row['Res_ID']
    ID = row['ID']
    dpka = row['dpka_stded']
    fasta = row['pth'].split('/')[-1].split('.')[0]
    esm_emb = torch.load(f'{emb_raw_pth}/{fasta}.pt', weights_only = False)
    pdb_df = PandasPdb().read_pdb(pdbpth)
    pdb_df_ = pdb_df.df['ATOM'].drop_duplicates(subset = ['residue_name', 'residue_number']).reset_index(drop=True)
    pdb_df_['index'] = pdb_df_.index
    ### pos_in_seq is the postion of residue of interest in seqence. 
    pos_in_seq = pdb_df_[pdb_df_['residue_number'] == resid]['index'].values[0]
    node_feat = esm_emb['representations'][layer][pos_in_seq].numpy().reshape(1, -1)
    y = torch.FloatTensor(np.array(dpka)).view(-1, 1)
    data = Data(x = node_feat, y = y, resname = ID)
    torch.save(data, f'{emb_ext_pth}/{ID}.pt')